In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [2]:
import pandas as pd
import json

df = pd.read_json(
    r"/home/user/Downloads/kentwood_2026_08_05.json",
    lines=True
)

# Convert list/dict columns into strings
df = df.apply(
    lambda col: col.map(
        lambda x: json.dumps(x, sort_keys=True)
        if isinstance(x, (list, dict))
        else x
    )
)

In [3]:
import pandas as pd
import re
import unicodedata

def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()

    # Remove accents
    text = "".join(
        c for c in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(c)
    )

    # Keep only letters, numbers, hyphens and spaces
    text = re.sub(r"[^a-z0-9\s-]", "", text)

    # Replace spaces with hyphens
    text = re.sub(r"\s+", "-", text)

    return text

def profile_name_matches(row):
    url = str(row["profile_url"]).lower()

    # Extract the slug after /our-advisors/ or /bio/
    m = re.search(r"/(?:our-advisors|bio)/([^/?]+)", url)
    if not m:
        return False

    slug = normalize(m.group(1))

    first = normalize(row["first_name"])
    last = normalize(row["last_name"])

    return first in slug and last in slug

invalid_profile_urls = df[
    ~df.apply(profile_name_matches, axis=1)
]

print(f"Profile URL name mismatches: {len(invalid_profile_urls)}")
display(invalid_profile_urls[["profile_url", "first_name", "last_name"]])

Profile URL name mismatches: 116


,profile_url,first_name,last_name
0,https://www.kentwood.com/bio/bkelly,Bob,Kelly
2,https://www.kentwood.com/bio/bgustafson,Brendan,Gustafson
3,https://www.kentwood.com/bio/Brandon,Brandon,Brennick
6,https://www.kentwood.com/bio/HarrisTeam,Brian and Jamie Harris,
8,https://www.kentwood.com/bio/brosen,Brian,Rosen
9,https://www.kentwood.com/bio/bridgetm,Bridget,Marx
10,https://www.kentwood.com/bio/BrinckerhoffEmery,Brinckerhoff Emery & Associates,
11,https://www.kentwood.com/bio/uhl,Bryan & Kris Uhl,
12,https://www.kentwood.com/bio/Jbuckley,Buckley,Team
14,https://www.kentwood.com/bio/Carolyn,Carolyn,O'Donnell


In [7]:
import re

email_pattern = re.compile(
    r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
)

invalid_emails = df[
    df["email"].notna() &
    (df["email"].str.strip() != "") &
    ~df["email"].astype(str).str.match(email_pattern)
]

print(f"Invalid emails: {len(invalid_emails)}")

display(invalid_emails[["profile_url", "email"]])

Invalid emails: 0


,profile_url,email


In [8]:
import re

description_issues = df[
    df["description"].notna() &
    (
        df["description"].str.contains(r"<[^>]+>", regex=True) |                  # HTML tags
        df["description"].ne(df["description"].str.strip()) |                     # Leading/trailing spaces
        df["description"].str.contains(r"\s{2,}", regex=True) |                   # Multiple spaces
        df["description"].str.contains(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", regex=True)  # Control chars
    )
]

print(f"Descriptions with issues: {len(description_issues)}")
display(description_issues[["profile_url", "description"]])

Descriptions with issues: 0


,profile_url,description


In [9]:
import re

first_name_issues = df[
    df["first_name"].isna() |
    (df["first_name"].str.strip() == "") |
    (df["first_name"] != df["first_name"].str.strip()) |
    df["first_name"].fillna("").str.contains(r"\s{2,}") |
    df["first_name"].fillna("").str.contains(r"[0-9@#$%^&*_=+<>/\\|{}\[\]~`]")
]

print(f"First names with issues: {len(first_name_issues)}")
display(first_name_issues[["profile_url", "first_name"]])

First names with issues: 6


,profile_url,first_name
10,https://www.kentwood.com/bio/BrinckerhoffEmery,Brinckerhoff Emery & Associates
11,https://www.kentwood.com/bio/uhl,Bryan & Kris Uhl
44,https://www.kentwood.com/bio/ginaandkara2,Gina Lorenzen & Kara Couzens
121,https://www.kentwood.com/bio/pattygreg,Patty Ryan Anton & Greg Card
140,https://www.kentwood.com/bio/larkin,Sean & Debbie Larkin
158,https://www.kentwood.com/bio/dutzerteam,Thomas & Rosanne Dutzer


In [10]:
import pandas as pd
import re

# URL regex
url_pattern = re.compile(
    r'^https?://'
    r'(([A-Za-z0-9-]+\.)+[A-Za-z]{2,})'
    r'(:\d+)?'
    r'(/[^\s]*)?$',
    re.IGNORECASE
)

issues = []

# Top-level URL columns
url_cols = ["profile_url", "website", "image_url"]

for col in url_cols:
    if col in df.columns:
        mask = (
            df[col].fillna("").astype(str).str.strip().ne("") &
            ~df[col].astype(str).str.strip().str.match(url_pattern)
        )

        if mask.any():
            temp = df.loc[mask, ["profile_url", col]].copy()
            temp.insert(0, "row_no", temp.index + 1)
            temp["column"] = col
            temp.rename(columns={col: "invalid_value"}, inplace=True)
            issues.append(temp)

# Check URLs inside social dictionary
if "social" in df.columns:
    social_fields = ["facebook_url", "twitter_url", "linkedin_url"]

    for idx, value in df["social"].items():
        if isinstance(value, dict):
            # facebook, twitter, linkedin
            for key in social_fields:
                url = str(value.get(key, "")).strip()
                if url and not url_pattern.match(url):
                    issues.append(pd.DataFrame({
                        "row_no": [idx + 1],
                        "profile_url": [df.at[idx, "profile_url"]],
                        "column": [f"social.{key}"],
                        "invalid_value": [url]
                    }))

            # other_urls (list)
            other_urls = value.get("other_urls", [])
            if isinstance(other_urls, list):
                for url in other_urls:
                    url = str(url).strip()
                    if url and not url_pattern.match(url):
                        issues.append(pd.DataFrame({
                            "row_no": [idx + 1],
                            "profile_url": [df.at[idx, "profile_url"]],
                            "column": ["social.other_urls"],
                            "invalid_value": [url]
                        }))

if issues:
    result = pd.concat(issues, ignore_index=True)
    display(result)
    print(f"\nTotal invalid URLs found: {len(result)}")
else:
    print("No invalid URL values found.")

No invalid URL values found.


In [12]:
df.profile_url.duplicated().sum()

np.int64(0)

In [13]:
df.shape

(195, 19)

In [14]:
import json

duplicate_count = (
    df.applymap(
        lambda x: json.dumps(x, sort_keys=True)
        if isinstance(x, (list, dict))
        else x
    )
    .duplicated()
    .sum()
)

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


/tmp/ipykernel_11794/600954025.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df.applymap(


In [15]:
req_cols = """first_name, middle_name, last_name, office_name, title, description, languages, image_url, address, city, state, country, zipcode, office_phone_numbers, agent_phone_numbers, email, website, social, profile_url"""

req_cols = [c.strip() for c in req_cols.split(",") if c.strip()]

missing = [c for c in req_cols if c not in df.columns]
extra = [c for c in df.columns if c not in req_cols]

print("Requirement columns:", len(req_cols))
print("File columns:", len(df.columns))
print("Missing:", missing)
print("Extra:", extra)
print("Order matches:", req_cols == list(df.columns))

Requirement columns: 19
File columns: 19
Missing: []
Extra: []
Order matches: False


In [16]:
import re

pattern = r'^\d+(\.\d+)?\s*م²$'

invalid = df[
    ~df["details"].fillna("").astype(str).str.match(pattern)
]

invalid.insert(0, "row_no", invalid.index + 1)

display(invalid[["row_no", "url", "details"]])

KeyError: 'details'

In [17]:
df.nunique()

profile_url             195
first_name              161
middle_name               8
last_name               173
image_url               195
office_name               8
address                   7
description             194
languages                14
social                  134
website                 146
email                     1
title                    54
country                   1
city                      5
zipcode                   6
state                     1
agent_phone_numbers     194
office_phone_numbers      7
dtype: int64

In [18]:
df.country.value_counts()

country
United States    195
Name: count, dtype: int64

In [19]:
df.state.value_counts()

state
CO    195
Name: count, dtype: int64

In [20]:
df[df.state == ""]

,profile_url,first_name,middle_name,last_name,image_url,office_name,address,description,languages,social,website,email,title,country,city,zipcode,state,agent_phone_numbers,office_phone_numbers


In [21]:
df.country.value_counts()

country
United States    195
Name: count, dtype: int64

In [22]:
import pandas as pd

df1 = df.replace(r'^\s*$', pd.NA, regex=True)

empty_columns = df1.columns[df1.isna().all()]

print("Completely empty columns:")
print(list(empty_columns))

Completely empty columns:
['email']


In [23]:
import pandas as pd

# Treat empty strings and whitespace-only values as NA
df1 = df.replace(r'^\s*$', pd.NA, regex=True)

# Requirement empty columns
req_empty_cols =  """
"""

req_empty_cols = [c.strip() for c in req_empty_cols.split(",") if c.strip()]

# File empty columns
file_empty_cols = list(df1.columns[df1.isna().all()])

matching = sorted(set(req_empty_cols) & set(file_empty_cols))
expected_not_empty = sorted(set(req_empty_cols) - set(file_empty_cols))
unexpected_empty = sorted(set(file_empty_cols) - set(req_empty_cols))

print("Requirement empty columns :", len(req_empty_cols))
print("File empty columns        :", len(file_empty_cols))
print("Matching                  :", len(matching))
print("Expected empty but not empty:", expected_not_empty)
print("Unexpected empty columns     :", unexpected_empty)

Requirement empty columns : 0
File empty columns        : 1
Matching                  : 0
Expected empty but not empty: []
Unexpected empty columns     : ['email']


In [25]:
import re

url_pattern = re.compile(
    r'^(https?://)'
    r'([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}'
    r'(/[^\s]*)?$'
)

invalid_url = df[
    df["website"].notna() &
    (df["website"].str.strip() != "") &
    ~df["website"].astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_url)}")

display(invalid_url[["website", "profile_url"]])

Invalid URLs: 0


,website,profile_url


In [27]:
df.city.value_counts()

city
Denver              165
Ft Collins           19
Boulder               8
Dillon                2
Colorado Springs      1
Name: count, dtype: int64

In [28]:
duplicates = df[df.duplicated(subset="profile_url", keep=False)]

print(f"Duplicate URLs: {len(duplicates)}")
display(duplicates.sort_values("profile_url"))

Duplicate URLs: 0


,profile_url,first_name,middle_name,last_name,image_url,office_name,address,description,languages,social,website,email,title,country,city,zipcode,state,agent_phone_numbers,office_phone_numbers


In [29]:
import re

url_pattern = re.compile(
    r'^(https?://)'
    r'([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}'
    r'(/[^\s]*)?$'
)

invalid_url = df[
    df["image_url"].notna() &                      # Exclude NaN
    (df["image_url"].str.strip() != "") &         # Exclude empty strings
    ~df["image_url"].astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_url)}")
display(invalid_url[["image_url"]])

Invalid URLs: 0


,image_url


In [31]:
import re

url_pattern = re.compile(
    r'^(https?://)'
    r'([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}'
    r'(/[^\s]*)?$'
)

invalid_url = df[
    ~df["image_url"].fillna("").astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_url)}")
display(invalid_url[["image_url"]])

Invalid URLs: 0


,image_url


In [32]:
empty_counts = (
    df.astype(str)
      .apply(lambda col: (col.str.strip() == "").sum())
)

print(empty_counts)

profile_url               0
first_name                0
middle_name             188
last_name                 8
image_url                 0
office_name               0
address                   0
description               1
languages                 0
social                    0
website                  30
email                   195
title                     0
country                   0
city                      0
zipcode                   0
state                     0
agent_phone_numbers       0
office_phone_numbers      0
dtype: int64


In [41]:
df[df.description == ""]

,profile_url,first_name,middle_name,last_name,image_url,office_name,address,description,languages,social,website,email,title,country,city,zipcode,state,agent_phone_numbers,office_phone_numbers
36,https://www.kentwood.com/bio/elise,Elise,,Marks,https://content.mediastg.net/Dynamic/RealEstate/company/624/account/539902/539902_06222023233404.jpg,Kentwood DTC,4949 South Niagara St 400,,[],{},https://elise.kentwood.com,,"Broker Associate, Realtor ®",United States,Denver,80237,CO,"[""(303) 947-0884""]","[""(303) 773-3399""]"


In [33]:
df.office_phone_numbers.value_counts()

office_phone_numbers
["(303) 773-3399"]    91
["(303) 331-1400"]    48
["(303) 820-2489"]    24
["(970) 300-1985"]    19
["303-785-3561"]      10
["970-300-1985"]       2
["719-259-3500"]       1
Name: count, dtype: int64

In [44]:
df[df['city'].isin(["Dillon", "Colorado Springs"])]


,profile_url,first_name,middle_name,last_name,image_url,office_name,address,description,languages,social,website,email,title,country,city,zipcode,state,agent_phone_numbers,office_phone_numbers
117,https://www.kentwood.com/bio/NoahBrooks,Noah,,Brooks,https://content.mediastg.net/Dynamic/RealEstate/company/624/account/6600175/6600175_04172026215716.jpg,Kentwood Mountain Properties,"265 Dillon Ridge Rd, Unit C/450","Noah and Danielle bring a dynamic partnership to the real estate experience. Noah, a former Director of the Ski Club Vail Freeride Team and a Warren Miller athlete, has a contagious enthusiasm for the outdoors that he shares with Danielle and their four boys. As residents of Clark, Colorado, they are often found exploring the mountains of North Routt County through hiking, skiing, snowmobiling, and biking. With over two decades of experience in real estate, Noah Brooks is a seasoned professional. He is a proud alumnus of Cherry Creek High School in Greenwood Village, Colorado, and the University of Colorado Boulder, where he majored in Economics with a Business Emphasis. While in college, he remodeled his first property to house the CU Ski Team in Winter Parksparking a passion that has since led him to remodel and sell more than 30 properties throughout Colorado. Noah also holds a Certification in Property Management from New York University. Danielle, a native of East Hampton, New York, studied Art and Photography at the University of Colorado Boulder. Her passion for healthy living led her and Noah to own and operate ranches in Colorado and Vermont, where they raised Himalayan yaks known for their lean, healthy meat. Through their ranch, Raddog Ranch, they even supplied meat to the Boston Red Sox. Danielle now owns and operates Neat and Nest, LLC in Clark, Colorado, specializing in the cleaning and maintenance of mountain homes. While not a licensed real estate agent, Danielle plays an integral role behind the scenes, offering valuable insight into home presentation, organization, and lifestyle considerations. Together, Noah and Danielle bring a wealth of knowledge in ranching and resort-area living, providing clients with a well-rounded perspective. Noahs extensive understanding of property values and features across marketsincluding Denver, Boulder, Golden, Conifer, Vail, Red Cliff, Steamboat Springs, Winter Park, Crested Butte, Clark, and Vermontallows him to expertly guide buyers and sellers through every step of the process. With Danielles eye for design and Noahs proven real estate expertise, clients benefit from both strategic guidance and thoughtful presentation. When you work with Noah, you gain not only a trusted real estate professional, but also the added support and insight that Danielle brings to the overall experience.",[],"{""facebook_url"": """", ""linkedin_url"": """", ""other_urls"": [""https://www.instagram.com/teambrookscolorado""], ""twitter_url"": """"}",https://NoahBrooks.kentwood.com,,"Broker Associate, Realtor®",United States,Dillon,80435,CO,"[""970-761-0678""]","[""970-300-1985""]"
124,https://www.kentwood.com/bio/PaulVandervort,Paul,,Vandervort,https://content.mediastg.net/Dynamic/RealEstate/company/624/account/6587716/6587716_03182026220550.jpg,Kentwood Southern Colorado,"1755 Telstar Drive, 3rd Floor","Paul Vandervort is a Broker Associate with Kentwood Real Estate, working with buyers, sellers, and investors across Colorado, with a strong focus on Colorado Springs. His work centers on helping clients navigate residential, luxury, and investment real estate with clear strategy, disciplined execution, and long-term perspective. Paul brings a capital-informed approach to real estate, supported by his background as a licensed mortgage originator and investor. This integrated experience allows him to help clients evaluate not only the property itself, but also how financing structure, timing, and risk affect the overall decision, particularly in new construction, luxury homes, and complex t

In [34]:
df.title.value_counts()

title
Broker Associate, Realtor®                                                106
Broker Associate, Realtor ®                                                13
Broker Associate                                                            7
Senior Commercial Advisor                                                   5
Broker Associate, Realtor®, CNE                                             4
Broker Associates, Realtors ®                                               3
Kentwood Real Estate Services                                               3
Broker Associates, Realtors®                                                3
Broker Associate, Realtor®, MBA                                             2
Broker Associate, Emeritus Status                                           2
Commercial Advisor                                                          2
Broker Associate, Realtor®, CRS                                             2
Broker Associate, GRI, CRS                                

In [35]:
df.country.value_counts()

country
United States    195
Name: count, dtype: int64

In [37]:
df[df.website == ""]

,profile_url,first_name,middle_name,last_name,image_url,office_name,address,description,languages,social,website,email,title,country,city,zipcode,state,agent_phone_numbers,office_phone_numbers
4,https://www.kentwood.com/bio/BrentGilles,Brent,,Gilles,https://content.mediastg.net/Dynamic/RealEstate/company/624/account/6318785/6318785_07312023034421.jpg,Kentwood Cherry Creek,215 St. Paul Street #200,"Brent Gilles understands the difference between finding clients a house and finding them a home. The fundamental distinction, he believes, is that while there may be lots of houses to consider in a given market, only a handful may truly satisfy the needs and desires of his client. For Brent, his reputation as a highly respected realtor is less about location, location, location, and more about experience, experience, experience. His 28 years spent working in real estate enables him to think outside the box, structuring deals and transactions that help his clients feel great about their experience and know they have made a solid decision. Whether he is showing someone 5 homes or 50, Brent brings enthusiasm and passion to every client. Brent prides himself on working diligently to make sure everyone -- family, friend, client or even someone hes just met -- is made to feel important. Putting others first, he believes, brings happiness and success to all, a key factor that brings him a steady base of referrals and repeat clients. He has experienced a wide spectrum of market conditions during his 28 years as a residential broker and imparts that vast knowledge to his clients. Brents enthusiasm for his profession stems from the belief that every day is different, and every transaction is unique, beginning with a clients wants, needs and ultimately their goals. He uses his strong knowledge of current market conditions to successfully navigate transactions in a professional and thoughtful manner. A Colorado native, Brent is well-versed in the unique neighborhoods that are part and parcel of the Denver metro area. His familiarity with Denver and surrounding suburbs helps him match clients with areas that best suit their needs, and he works tirelessly to help them find the perfect home that will serve them now and in the future. His clients appreciate his candor, and he will not hesitate to point out aspects of a home they may not recognize, discussing pros and cons to ensure they have as much information as possible. Im a real people person and I take the time to listen and understand what is best for my client, Brent says. My goal is for every client to understand how we can together successfully meet their expectations. My clients value my honesty and know I will never encourage them to consider a property that is not aligned with their wants, needs and goals. An avid skier, Brent clicked into his first pair of skis at age three and could not get enough of the sport and gratitude for the surroundings. Eventually the passion for skiing and particularly moguls led to competing for the Winter Park Freestyle Ski Team. He also enjoys golf, tennis, volleyball, pickleball and loves to take advantage of Colorados endless outdoor activities!",[],{},,,"Broker Associate, Realtor®",United States,Denver,80206,CO,"[""720-217-8172""]","[""(303) 331-1400""]"
15,https://www.kentwood.com/bio/CarynGillman,Caryn,,Gillman,https://content.mediastg.net/Dynamic/RealEstate/company/624/account/6518158/6518158_02052026183143.png,Kentwood Northern Properties,2950 E Harmony Road #210,"You may know something about me, you may know nothing. Here's a short intro on me and I look forward to getting to know YOU, what you love and what is important to YOU! Meet Caryn :Proud mom of two energetic boys, a loving wife, outdoor enthusiast and dedicated Realtor. Caryn has 13 years of experience in Real Estate both locally and internationally, she is passionate about helping people and developing authentic connections built on transparency and trust. Caryn is known for her honesty,

In [31]:
df.image_url.value_counts()

image_url
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6365079/6365079_11292023150828.jpg     1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181848/6181848_06222023011629.jpeg    1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181847/6181847_06222023011312.jpeg    1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181796/6181796_03272023142000.jpg     1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6568451/6568451_02062026170958.jpg     1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181661/6181661_06162023213933.jpeg    1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6593859/6593859_03232026123921.jpg     1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181658/6181658_01082025211247.jpg     1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6387774/6387774_10242023194810.jpg     1
https://c

In [32]:
df.image_url.duplicated().sum()

np.int64(0)

In [33]:
df.columns

Index(['profile_url', 'first_name', 'middle_name', 'last_name', 'image_url',
       'office_name', 'address', 'description', 'languages', 'social',
       'website', 'email', 'title', 'country', 'city', 'zipcode', 'state',
       'agent_phone_numbers', 'office_phone_numbers'],
      dtype='object')

In [34]:
df.profile_url.duplicated().sum()

np.int64(0)

In [35]:
# Find records with duplicate image_url
duplicates = df[df.duplicated(subset=["image_url"], keep=False)] \
    .reset_index(names="row_no") \
    .sort_values("image_url")

print(f"Duplicate image URLs: {duplicates['image_url'].nunique()}")
print(duplicates[["row_no", "profile_url", "image_url"]])

Duplicate image URLs: 0
Empty DataFrame
Columns: [row_no, profile_url, image_url]
Index: []


In [36]:
duplicate_records = (
    df.reset_index(names="row_no")
      .loc[lambda x: x.duplicated(subset=["image_url"], keep=False)]
      .sort_values("image_url")
)

duplicate_records

,row_no,profile_url,first_name,middle_name,last_name,image_url,office_name,address,description,languages,social,website,email,title,country,city,zipcode,state,agent_phone_numbers,office_phone_numbers


In [37]:
print(df["image_url"].isna().sum())
print(df["image_url"].eq("").sum())   # empty strings

0
0


In [38]:
print(df["image_url"].nunique(dropna=True))

172


In [39]:
print(df["image_url"].value_counts(dropna=False).head(30))

image_url
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6365079/6365079_11292023150828.jpg     1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181848/6181848_06222023011629.jpeg    1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181847/6181847_06222023011312.jpeg    1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181796/6181796_03272023142000.jpg     1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6568451/6568451_02062026170958.jpg     1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181661/6181661_06162023213933.jpeg    1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6593859/6593859_03232026123921.jpg     1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181658/6181658_01082025211247.jpg     1
https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6387774/6387774_10242023194810.jpg     1
https://c

In [40]:
vc = df["image_url"].value_counts()

print(vc[vc > 1])

Series([], Name: count, dtype: int64)


In [13]:
import ast
import re
import pandas as pd

phone_cols = ["agent_phone_numbers", "office_phone_numbers"]

# Allowed: digits, spaces, +, -, (, )
phone_pattern = re.compile(r'^[\d\s()+-]+$')

for col in phone_cols:
    invalid_rows = []

    for idx, val in df[col].items():

        if pd.isna(val) or val == "":
            continue

        # Convert string representation of list to list
        if isinstance(val, str):
            try:
                phones = ast.literal_eval(val)
            except Exception:
                phones = [val]
        else:
            phones = val

        if not isinstance(phones, list):
            phones = [phones]

        for phone in phones:
            phone = str(phone).strip()

            # Ignore empty entries
            if phone == "":
                continue

            # Check invalid characters
            if not phone_pattern.fullmatch(phone):
                invalid_rows.append(idx)
                break

    if invalid_rows:
        print(f"\n{col}: {len(invalid_rows)} invalid records")
        print(df.loc[invalid_rows, [col]])
    else:
        print(f"\n{col}: No invalid phone numbers found.")

KeyError: 'agent_phone_numbers'

In [42]:
df[df.middle_name == ""]

,profile_url,first_name,middle_name,last_name,image_url,office_name,address,description,languages,social,website,email,title,country,city,zipcode,state,agent_phone_numbers,office_phone_numbers
0,https://www.robertsbrothers.com/bio/cherishtrocchio,Cherish,,Trocchio,https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6365079/6365079_11292023150828.jpg,,900 Bienville Blvd.,"Cherish Trocchio, a distinguished member of The Fox Team, specializes in real estate on Dauphin Island, Alabama. Beyond being a professional in this coastal community, she proudly calls it home, infusing her work with a deep understanding and passion for the area. Having personally relocated to Dauphin Island in 2020, Cherish navigates the logistics of island living for her clients. She serves as a valuable source of information on island services, businesses, and amenities, ensuring a seamless experience for those transitioning to this beautiful locale. With over two decades of entrepreneurial success, Cherish's transition into real estate is marked by her extensive business background, offering a unique perspective on customer service and an unparalleled work ethic. She is more than just a real estate agent, she stands as a local expert, armed with an intimate knowledge of market trends and a commitment to helping clients achieve their goals both before and after the sale. Clients benefit from Cherish Trocchio's personalized approach, where she exceeds expectations to understand their needs. Her unwavering dedication to client satisfaction establishes her as a trusted and reliable partner in the Dauphin Island real estate landscape.","[""English""]",{},https://cherishtrocchio.robertsbrothers.com,,Real Estate Assistant,United States,Dauphin Island,36528,AL,"[""330-524-8993""]",[]
2,https://www.robertsbrothers.com/bio/chrisclarke,Chris,,Clarke,https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181847/6181847_06222023011312.jpeg,,3601 Spring Hill Business Park Ste 101,"If theres one thing that keeps Chris Clarke both passionate and positive about real estate, its you. Having the opportunity connect with you, take care of your unique needs, help you navigate the ever-changing market, and leave you feeling confident about one of your biggest decisions, allows him to give of part of himself back to his community. Giving back is the best part. Chris is a respected top agent and multi-million-dollar producer with over 25 years of sales experience. He takes pride in being a mentor to other successful agents at Roberts Brothers and is recognized as a pacesetter by the company. Born and raised here in Mobile, Alabama, Chris attended St. Pius, McGill-Toolen, and Spring Hill College. His integrity and professionalism are paramount to him, and he attributes this to his education and by the example of his parents, who worked hard and always taught him to do the right thing. Chris will employ that same philosophy for you. Chris loves where he lives. With a genuine appreciation of Mobiles unique history, he has overseen the historic renovation of eleven of his personal properties in the city. He is passionate about his roots and takes a huge interest in The Original Mardi Gras. Chris does CrossFit, Spartan Racing, Tough Mudder and many local 5K and 10K races. This, combined with a spiritual life that is very important to him, keeps him disciplined and balanced. Whatever your motivation for buying and selling, Chris is ready to take this journey with you, which he considers it a privilege, never to be take for granted.","[""English""]",{},https://chrisclarke.robertsbrothers.com,,Sales Associate - REALTOR®,United States,Mobile,36608,AL,"[""251-709-1677""]",[]
3,https://www.robertsbrothers.com/bio/chrissimoore,Chrissi,,Moore,https://content.mediastg.net/Dynamic/RealEstate/company/772/account/6181796/6181796_03272023142000.jpg,,"6721 Grelot Road, Suite A","Why accept less when you expect ""Moore"", call Chrissi today! Chrissi is your ""Next Gen"

In [38]:
df[df.zipcode == ""]

,profile_url,first_name,middle_name,last_name,image_url,office_name,address,description,languages,social,website,email,title,country,city,zipcode,state,agent_phone_numbers,office_phone_numbers


In [39]:
df.zipcode.value_counts()

zipcode
80237    83
80206    82
80528    19
80302     8
80435     2
80920     1
Name: count, dtype: int64

In [40]:
import pandas as pd
import re
import unicodedata

def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()

    # Remove accents
    text = "".join(
        c for c in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(c)
    )

    # Keep only letters, numbers, hyphens and spaces
    text = re.sub(r"[^a-z0-9\s-]", "", text)

    # Replace spaces with hyphens
    text = re.sub(r"\s+", "-", text)

    return text

def profile_name_matches(row):
    url = str(row["profile_url"]).lower()

    # Extract the slug after /our-advisors/ or /bio/
    m = re.search(r"/(?:our-advisors|bio)/([^/?]+)", url)
    if not m:
        return False

    slug = normalize(m.group(1))

    first = normalize(row["first_name"])
    last = normalize(row["last_name"])

    return first in slug and last in slug

invalid_profile_urls = df[
    ~df.apply(profile_name_matches, axis=1)
]

print(f"Profile URL name mismatches: {len(invalid_profile_urls)}")
display(invalid_profile_urls[["profile_url", "first_name", "last_name"]])

Profile URL name mismatches: 116


,profile_url,first_name,last_name
0,https://www.kentwood.com/bio/bkelly,Bob,Kelly
2,https://www.kentwood.com/bio/bgustafson,Brendan,Gustafson
3,https://www.kentwood.com/bio/Brandon,Brandon,Brennick
6,https://www.kentwood.com/bio/HarrisTeam,Brian and Jamie Harris,
8,https://www.kentwood.com/bio/brosen,Brian,Rosen
9,https://www.kentwood.com/bio/bridgetm,Bridget,Marx
10,https://www.kentwood.com/bio/BrinckerhoffEmery,Brinckerhoff Emery & Associates,
11,https://www.kentwood.com/bio/uhl,Bryan & Kris Uhl,
12,https://www.kentwood.com/bio/Jbuckley,Buckley,Team
14,https://www.kentwood.com/bio/Carolyn,Carolyn,O'Donnell
